# Proyecto de Analítica de Clientes — Versión corregida

**Caso:** empresa de servicios digitales por suscripción con aumento de abandono de clientes.

**Objetivo general:** comprender el comportamiento de los clientes, identificar factores asociados al abandono y preparar un flujo de datos limpio, validado y transformado para análisis y futuros modelos predictivos.

Esta versión corrige los puntos observados en la retroalimentación:

- Se incorporan filtros avanzados, agrupaciones múltiples y joins.
- Cada visualización incluye interpretación analítica.
- Las transformaciones se aplican realmente al dataset, no solo se definen funciones.
- La limpieza trata nulos, duplicados, valores inválidos, tipos de datos y outliers.
- Se agregan validaciones de integridad, esquema y procedencia.
- El código se organiza en funciones con docstrings, modularidad y manejo de excepciones.

## 1. Importación de librerías y configuración

Se importan las librerías principales para manipulación, validación, visualización y preprocesamiento.  
La configuración de visualización permite generar gráficos legibles para el informe.

In [ ]:
import os
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

RANDOM_STATE = 42

: 

## 2. Carga de datos con manejo de excepciones

La carga de datos se encapsula en una función para mejorar la reutilización y controlar errores.  
Primero se intenta cargar el archivo local `dataset_clientes.csv`. Si no existe, se utiliza la URL del repositorio.

In [ ]:
def cargar_datos(ruta_local="dataset_clientes.csv", url=None):
    """
    Carga un dataset desde una ruta local o desde una URL.

    Parámetros
    ----------
    ruta_local : str
        Nombre o ruta del archivo CSV local.
    url : str | None
        URL alternativa para cargar el archivo si no existe localmente.

    Retorna
    -------
    pd.DataFrame
        DataFrame cargado.

    Lanza
    -----
    FileNotFoundError
        Si no se encuentra archivo local ni URL válida.
    ValueError
        Si el archivo existe pero no puede leerse como CSV.
    """
    try:
        if Path(ruta_local).exists():
            df = pd.read_csv(ruta_local)
            print(f"Dataset cargado desde archivo local: {ruta_local}")
        elif url is not None:
            df = pd.read_csv(url)
            print("Dataset cargado desde URL.")
        else:
            raise FileNotFoundError("No se encontró el archivo local y no se entregó URL alternativa.")
        print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
        return df
    except pd.errors.ParserError as error:
        raise ValueError("El archivo no tiene un formato CSV válido.") from error
    except Exception as error:
        raise RuntimeError(f"Error al cargar los datos: {error}") from error

url_github = "https://raw.githubusercontent.com/magomezn-debug/dScience_001D/main/data/dataset_clientes.csv"
df_original = cargar_datos("dataset_clientes.csv", url_github)
df_original.head()

## 3. Procedencia e integridad del archivo

La rúbrica exige verificar integridad y procedencia.  
Para ello se calcula una suma de verificación SHA-256 del archivo. Esto permite demostrar si el archivo usado en el análisis cambió o no cambió.

In [ ]:
def calcular_sha256(ruta_archivo):
    """
    Calcula el hash SHA-256 de un archivo para verificar integridad.
    """
    sha256 = hashlib.sha256()
    with open(ruta_archivo, "rb") as archivo:
        for bloque in iter(lambda: archivo.read(8192), b""):
            sha256.update(bloque)
    return sha256.hexdigest()

if Path("dataset_clientes.csv").exists():
    hash_dataset = calcular_sha256("dataset_clientes.csv")
    print("SHA-256 del dataset:", hash_dataset)
else:
    print("No se pudo calcular hash porque el archivo no está en ruta local.")

## 4. Diccionario de datos esperado y validación de esquema

Antes de limpiar, se valida que el dataset tenga las columnas esperadas y tipos compatibles con el caso de estudio.
Esto evita trabajar con columnas mal escritas, ausentes o mal interpretadas.

In [ ]:
columnas_esperadas = {
    "id_cliente": "int64",
    "fecha_registro": "object",
    "edad": "int64",
    "genero": "object",
    "region": "object",
    "estado_civil": "object",
    "ingreso_mensual": "float64",
    "gasto_mensual": "float64",
    "deuda_total": "float64",
    "score_crediticio": "float64",
    "antiguedad_meses": "int64",
    "frecuencia_compra": "int64",
    "ultima_compra_dias": "int64",
    "uso_app": "object",
    "tipo_plan": "object",
    "num_productos": "int64",
    "tiene_tarjeta_credito": "int64",
    "canal_registro": "object",
    "dia_semana_registro": "object",
    "hora_registro": "int64",
    "codigo_postal": "int64",
    "abandono": "int64"
}

def validar_esquema(df, columnas_esperadas):
    """
    Valida presencia de columnas esperadas y reporta columnas faltantes o adicionales.
    """
    esperadas = set(columnas_esperadas.keys())
    actuales = set(df.columns)
    faltantes = sorted(esperadas - actuales)
    adicionales = sorted(actuales - esperadas)

    print("Columnas esperadas:", len(esperadas))
    print("Columnas actuales:", len(actuales))
    print("Columnas faltantes:", faltantes if faltantes else "Ninguna")
    print("Columnas adicionales:", adicionales if adicionales else "Ninguna")

    if faltantes:
        raise ValueError("El dataset no contiene todas las columnas requeridas.")

validar_esquema(df_original, columnas_esperadas)
df_original.info()

## 5. Exploración inicial del dataset

Esta etapa no limpia todavía. Primero se inspecciona el estado real de los datos: tamaño, tipos, nulos, duplicados, categorías y valores numéricos extremos.

In [ ]:
def resumen_inicial(df):
    """
    Genera un resumen inicial de tamaño, nulos, duplicados y tipos de datos.
    """
    resumen = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "nulos_%": (df.isna().mean() * 100).round(2),
        "unicos": df.nunique()
    })
    print(f"Filas totales: {df.shape[0]:,}")
    print(f"Columnas totales: {df.shape[1]}")
    print(f"Duplicados exactos: {df.duplicated().sum():,}")
    print(f"IDs duplicados: {df['id_cliente'].duplicated().sum():,}")
    return resumen

resumen_inicial(df_original)

In [ ]:
resumen_columnas = resumen_inicial(df_original)
resumen_columnas

## 6. Validación de dominios y reglas de negocio

Se revisan reglas mínimas de coherencia:

- `abandono` y `tiene_tarjeta_credito` deben ser binarias: 0/1.
- `edad` debe estar dentro de un rango posible.
- `hora_registro` debe estar entre 0 y 23.
- variables monetarias no deben ser negativas.
- `score_crediticio` debe estar en un rango razonable.

In [ ]:
def validar_reglas_negocio(df):
    """
    Valida reglas básicas de calidad e integridad de datos.
    Retorna un DataFrame con la cantidad de registros problemáticos por regla.
    """
    reglas = {
        "edad_fuera_rango": (~df["edad"].between(18, 100)).sum(),
        "hora_fuera_rango": (~df["hora_registro"].between(0, 23)).sum(),
        "abandono_no_binario": (~df["abandono"].isin([0, 1])).sum(),
        "tarjeta_no_binaria": (~df["tiene_tarjeta_credito"].isin([0, 1])).sum(),
        "ingreso_negativo": (df["ingreso_mensual"] < 0).sum(),
        "gasto_negativo": (df["gasto_mensual"] < 0).sum(),
        "deuda_negativa": (df["deuda_total"] < 0).sum(),
        "score_fuera_rango": (~df["score_crediticio"].dropna().between(0, 1000)).sum()
    }
    return pd.DataFrame.from_dict(reglas, orient="index", columns=["cantidad"])

validacion_negocio = validar_reglas_negocio(df_original)
validacion_negocio

## 7. Limpieza de datos

La limpieza se aplica sobre una copia del dataset original para mantener trazabilidad.  
Decisiones técnicas:

- **Duplicados:** se eliminan duplicados exactos porque representan el mismo registro repetido.
- **Fechas:** se convierten a tipo datetime para permitir análisis temporal.
- **Valores negativos en variables monetarias:** se reemplazan por `NaN`, porque no son válidos para ingreso, gasto o deuda en este caso.
- **Nulos numéricos:** se imputan con mediana porque existen valores extremos y la mediana es más robusta.
- **Nulos categóricos:** se imputan con moda.
- **Categorías:** se normaliza texto para evitar categorías duplicadas por formato.
- **Outliers:** se tratan con winsorización por IQR en variables monetarias/score para reducir impacto extremo sin eliminar clientes.

In [ ]:
def normalizar_texto_categoria(serie):
    """
    Normaliza texto categórico: elimina espacios externos y deja formato título.
    """
    return serie.astype(str).str.strip().str.title()


def limpiar_datos(df):
    """
    Aplica un flujo de limpieza completo sobre el dataset de clientes.

    Pasos:
    1. Elimina duplicados exactos.
    2. Convierte fecha_registro a datetime.
    3. Normaliza columnas categóricas.
    4. Convierte valores monetarios negativos en NaN.
    5. Imputa nulos numéricos con mediana.
    6. Imputa nulos categóricos con moda.

    Retorna
    -------
    pd.DataFrame
        Dataset limpio.
    dict
        Registro de decisiones aplicadas.
    """
    df_limpio = df.copy()
    log = {}

    # 1. Duplicados exactos
    log["duplicados_eliminados"] = int(df_limpio.duplicated().sum())
    df_limpio = df_limpio.drop_duplicates().copy()

    # 2. Fecha
    df_limpio["fecha_registro"] = pd.to_datetime(df_limpio["fecha_registro"], errors="coerce")
    log["fechas_invalidas"] = int(df_limpio["fecha_registro"].isna().sum())

    # 3. Normalización de categorías
    cat_cols = [
        "genero", "region", "estado_civil", "uso_app",
        "tipo_plan", "canal_registro", "dia_semana_registro"
    ]
    for col in cat_cols:
        df_limpio[col] = normalizar_texto_categoria(df_limpio[col])

    # 4. Valores negativos inválidos
    cols_no_negativas = ["ingreso_mensual", "gasto_mensual", "deuda_total"]
    for col in cols_no_negativas:
        negativos = df_limpio[col] < 0
        log[f"{col}_negativos_a_nan"] = int(negativos.sum())
        df_limpio.loc[negativos, col] = np.nan

    # 5. Imputación numérica con mediana
    num_cols_imputar = ["ingreso_mensual", "gasto_mensual", "deuda_total", "score_crediticio"]
    for col in num_cols_imputar:
        nulos_antes = int(df_limpio[col].isna().sum())
        mediana = df_limpio[col].median()
        df_limpio[col] = df_limpio[col].fillna(mediana)
        log[f"{col}_nulos_imputados"] = nulos_antes
        log[f"{col}_mediana_usada"] = float(mediana)

    # 6. Imputación categórica con moda
    for col in cat_cols:
        nulos_antes = int(df_limpio[col].isna().sum())
        if nulos_antes > 0:
            moda = df_limpio[col].mode(dropna=True)[0]
            df_limpio[col] = df_limpio[col].fillna(moda)
            log[f"{col}_nulos_imputados"] = nulos_antes
            log[f"{col}_moda_usada"] = moda

    return df_limpio, log


df_limpio, log_limpieza = limpiar_datos(df_original)
pd.DataFrame.from_dict(log_limpieza, orient="index", columns=["valor"])

## 8. Tratamiento de outliers mediante IQR

Los outliers pueden ser datos reales y relevantes, por eso no se eliminan automáticamente.  
Para variables económicas se usa winsorización: los valores extremos se limitan al rango aceptable calculado por IQR. Esta técnica reduce el impacto de valores extremos sin perder registros completos.

In [ ]:
def winsorizar_iqr(df, columnas, factor=1.5):
    """
    Limita valores extremos usando la regla IQR.

    Para cada columna:
    limite inferior = Q1 - factor * IQR
    limite superior = Q3 + factor * IQR

    Retorna el DataFrame ajustado y un reporte de outliers detectados.
    """
    df_out = df.copy()
    reporte = []

    for col in columnas:
        q1 = df_out[col].quantile(0.25)
        q3 = df_out[col].quantile(0.75)
        iqr = q3 - q1
        limite_inf = q1 - factor * iqr
        limite_sup = q3 + factor * iqr

        outliers = ((df_out[col] < limite_inf) | (df_out[col] > limite_sup)).sum()
        reporte.append({
            "variable": col,
            "q1": q1,
            "q3": q3,
            "iqr": iqr,
            "limite_inferior": limite_inf,
            "limite_superior": limite_sup,
            "outliers_detectados": int(outliers)
        })

        df_out[col] = df_out[col].clip(lower=limite_inf, upper=limite_sup)

    return df_out, pd.DataFrame(reporte)

cols_outliers = ["ingreso_mensual", "gasto_mensual", "deuda_total", "score_crediticio"]
df_limpio, reporte_outliers = winsorizar_iqr(df_limpio, cols_outliers)
reporte_outliers

## 9. Validación posterior a limpieza

La limpieza debe comprobarse.  
Se verifica que no queden nulos en variables relevantes, que no existan duplicados exactos y que las reglas de negocio críticas estén corregidas.

In [ ]:
validacion_post_limpieza = pd.DataFrame({
    "nulos": df_limpio.isna().sum(),
    "tipo": df_limpio.dtypes.astype(str),
    "unicos": df_limpio.nunique()
})

print("Duplicados exactos posteriores:", df_limpio.duplicated().sum())
print("Filas finales:", df_limpio.shape[0])
validacion_post_limpieza

In [ ]:
validar_reglas_negocio(df_limpio)

## 10. Ingeniería de características

La ingeniería de características transforma variables originales en variables con sentido analítico y de negocio.  
En este caso se crean variables relacionadas con presión financiera, antigüedad, uso y comportamiento del cliente.

Estas variables no solo son cálculos: representan hipótesis de negocio sobre el abandono.

In [ ]:
def crear_caracteristicas(df):
    """
    Crea variables derivadas con sentido de negocio para el análisis de abandono.

    Variables creadas:
    - ratio_gasto_ingreso: proporción del ingreso comprometido en gasto mensual.
    - ratio_deuda_ingreso: presión de deuda respecto del ingreso.
    - antiguedad_anios: antigüedad expresada en años.
    - cliente_reciente: cliente con menos de 12 meses de antigüedad.
    - cliente_inactivo: cliente sin compras recientes en más de 180 días.
    - registro_fin_semana: registro realizado sábado o domingo.
    - franja_horaria: clasificación de la hora de registro.
    """
    df_feat = df.copy()

    ingreso_seguro = df_feat["ingreso_mensual"].replace(0, np.nan)
    df_feat["ratio_gasto_ingreso"] = (df_feat["gasto_mensual"] / ingreso_seguro).replace([np.inf, -np.inf], np.nan)
    df_feat["ratio_deuda_ingreso"] = (df_feat["deuda_total"] / ingreso_seguro).replace([np.inf, -np.inf], np.nan)

    df_feat["ratio_gasto_ingreso"] = df_feat["ratio_gasto_ingreso"].fillna(df_feat["ratio_gasto_ingreso"].median())
    df_feat["ratio_deuda_ingreso"] = df_feat["ratio_deuda_ingreso"].fillna(df_feat["ratio_deuda_ingreso"].median())

    df_feat["antiguedad_anios"] = df_feat["antiguedad_meses"] / 12
    df_feat["cliente_reciente"] = (df_feat["antiguedad_meses"] < 12).astype(int)
    df_feat["cliente_inactivo"] = (df_feat["ultima_compra_dias"] > 180).astype(int)
    df_feat["registro_fin_semana"] = df_feat["dia_semana_registro"].isin(["Sabado", "Domingo"]).astype(int)

    condiciones = [
        df_feat["hora_registro"].between(0, 5),
        df_feat["hora_registro"].between(6, 11),
        df_feat["hora_registro"].between(12, 17),
        df_feat["hora_registro"].between(18, 23)
    ]
    valores = ["Madrugada", "Manana", "Tarde", "Noche"]
    df_feat["franja_horaria"] = np.select(condiciones, valores, default="Sin dato")

    return df_feat


df_final = crear_caracteristicas(df_limpio)
df_final[["ingreso_mensual", "gasto_mensual", "ratio_gasto_ingreso", "deuda_total", "ratio_deuda_ingreso", "antiguedad_anios"]].head()

## 11. Manipulación avanzada con Pandas: filtros, agrupaciones múltiples y joins

La rúbrica exige operaciones de manipulación más allá de consultas simples.  
Por eso se incluyen filtros con múltiples condiciones, agrupaciones por más de una variable y joins entre tablas derivadas.

In [ ]:
# Filtro avanzado: clientes con abandono, alto endeudamiento relativo y baja frecuencia de compra
clientes_riesgo = df_final[
    (df_final["abandono"] == 1) &
    (df_final["ratio_deuda_ingreso"] > df_final["ratio_deuda_ingreso"].quantile(0.75)) &
    (df_final["frecuencia_compra"] <= df_final["frecuencia_compra"].median())
].copy()

print("Clientes detectados en segmento de riesgo:", clientes_riesgo.shape[0])
clientes_riesgo.head()

In [ ]:
# Agrupación múltiple: abandono por tipo de plan y uso de app
reporte_plan_uso = (
    df_final
    .groupby(["tipo_plan", "uso_app"])
    .agg(
        clientes=("id_cliente", "count"),
        tasa_abandono=("abandono", "mean"),
        ingreso_promedio=("ingreso_mensual", "mean"),
        gasto_promedio=("gasto_mensual", "mean"),
        ratio_gasto_promedio=("ratio_gasto_ingreso", "mean")
    )
    .reset_index()
    .sort_values("tasa_abandono", ascending=False)
)

reporte_plan_uso

In [ ]:
# Creación de tablas derivadas para demostrar joins analíticos
clientes_base = df_final[["id_cliente", "edad", "genero", "region", "estado_civil", "tipo_plan", "abandono"]].copy()
comportamiento = df_final[["id_cliente", "gasto_mensual", "frecuencia_compra", "ultima_compra_dias", "uso_app", "cliente_inactivo"]].copy()
finanzas = df_final[["id_cliente", "ingreso_mensual", "deuda_total", "score_crediticio", "ratio_gasto_ingreso", "ratio_deuda_ingreso"]].copy()

# Join 1: clientes + comportamiento
clientes_comportamiento = clientes_base.merge(comportamiento, on="id_cliente", how="left")

# Join 2: resultado anterior + finanzas
clientes_integrado = clientes_comportamiento.merge(finanzas, on="id_cliente", how="left")

print("Tabla base:", clientes_base.shape)
print("Tabla comportamiento:", comportamiento.shape)
print("Tabla finanzas:", finanzas.shape)
print("Tabla integrada:", clientes_integrado.shape)
clientes_integrado.head()

**Interpretación de manipulación avanzada:**  
El filtro avanzado permite identificar clientes abandonados con señales de riesgo financiero y bajo comportamiento de compra.  
La agrupación múltiple permite analizar cómo cambia el abandono según plan y uso de aplicación.  
Los joins demuestran integración de fuentes, simulando un escenario real donde datos demográficos, financieros y de comportamiento pueden venir separados.

## 12. Reportes estadísticos con conclusiones

Los reportes no deben limitarse a tablas: deben generar hallazgos.  
A continuación se calculan estadísticos y se interpretan en relación con el abandono.

In [ ]:
resumen_abandono = (
    df_final
    .groupby("abandono")
    .agg(
        clientes=("id_cliente", "count"),
        ingreso_mediano=("ingreso_mensual", "median"),
        gasto_mediano=("gasto_mensual", "median"),
        deuda_mediana=("deuda_total", "median"),
        score_mediano=("score_crediticio", "median"),
        ratio_gasto_mediano=("ratio_gasto_ingreso", "median"),
        ultima_compra_mediana=("ultima_compra_dias", "median")
    )
    .reset_index()
)

resumen_abandono["tasa_clientes_%"] = (resumen_abandono["clientes"] / resumen_abandono["clientes"].sum() * 100).round(2)
resumen_abandono

**Conclusión del reporte estadístico:**  
La comparación entre clientes que abandonan y no abandonan permite identificar diferencias en variables económicas, crediticias y de comportamiento. Si el grupo con abandono presenta mayor deuda relativa, menor score o mayor cantidad de días desde la última compra, estas variables se convierten en candidatas relevantes para explicar el churn y alimentar modelos predictivos posteriores.

## 13. Visualizaciones con interpretación

Cada gráfico se acompaña de una lectura analítica. Esto corrige el problema de entregar gráficos sin comunicación de resultados.

In [ ]:
plt.figure(figsize=(6, 4))
tasa_abandono = df_final["abandono"].value_counts(normalize=True).sort_index() * 100
plt.bar(tasa_abandono.index.astype(str), tasa_abandono.values)
plt.title("Distribución porcentual de abandono")
plt.xlabel("Abandono (0 = No, 1 = Sí)")
plt.ylabel("Porcentaje")
for i, valor in enumerate(tasa_abandono.values):
    plt.text(i, valor + 1, f"{valor:.1f}%", ha="center")
plt.ylim(0, 100)
plt.show()

print("Hallazgo: este gráfico permite dimensionar el problema de negocio. Si la proporción de abandono es relevante, justificaría acciones preventivas de retención.")

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=reporte_plan_uso, x="tipo_plan", y="tasa_abandono", hue="uso_app")
plt.title("Tasa de abandono por tipo de plan y uso de app")
plt.xlabel("Tipo de plan")
plt.ylabel("Tasa de abandono")
plt.legend(title="Uso app")
plt.show()

print("Hallazgo: comparar plan y uso de app permite detectar segmentos específicos. Una tasa más alta en usuarios de bajo uso puede indicar menor percepción de valor del servicio.")

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=df_final, x="abandono", y="ratio_gasto_ingreso")
plt.title("Ratio gasto/ingreso según abandono")
plt.xlabel("Abandono")
plt.ylabel("Gasto mensual / Ingreso mensual")
plt.show()

print("Hallazgo: el ratio gasto/ingreso mide presión financiera. Si los clientes que abandonan tienen ratio más alto, podría existir relación entre carga económica y cancelación.")

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=df_final, x="abandono", y="score_crediticio")
plt.title("Score crediticio según abandono")
plt.xlabel("Abandono")
plt.ylabel("Score crediticio")
plt.show()

print("Hallazgo: el score crediticio puede representar estabilidad financiera. Diferencias entre grupos pueden aportar señales para modelos predictivos.")

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=df_final, x="abandono", y="ultima_compra_dias")
plt.title("Días desde última compra según abandono")
plt.xlabel("Abandono")
plt.ylabel("Días desde última compra")
plt.show()

print("Hallazgo: una mayor cantidad de días desde la última compra puede reflejar menor actividad o desconexión del cliente antes del abandono.")

In [ ]:
plt.figure(figsize=(8, 5))
correlacion = df_final[[
    "edad", "ingreso_mensual", "gasto_mensual", "deuda_total",
    "score_crediticio", "antiguedad_meses", "frecuencia_compra",
    "ultima_compra_dias", "num_productos", "ratio_gasto_ingreso",
    "ratio_deuda_ingreso", "abandono"
]].corr(numeric_only=True)

sns.heatmap(correlacion, cmap="coolwarm", center=0, annot=False)
plt.title("Matriz de correlación de variables numéricas")
plt.show()

print("Hallazgo: la matriz permite identificar relaciones lineales preliminares entre variables. No prueba causalidad, pero orienta selección de variables para análisis y modelado.")

## 14. Preprocesamiento para modelado mediante Pipeline

Aunque esta entrega se centra en manipulación, limpieza y transformación, dejar un pipeline preparado demuestra un flujo reproducible para modelos posteriores.

Decisiones:

- Variables numéricas: imputación mediana + estandarización.
- Variables categóricas: imputación moda + OneHotEncoder.
- Se excluyen identificadores y variables que no deben entrar directamente al modelo, como `id_cliente` y `fecha_registro`.

In [ ]:
target = "abandono"
columnas_excluir = ["id_cliente", "fecha_registro", "codigo_postal", target]

X = df_final.drop(columns=columnas_excluir)
y = df_final[target]

num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Variables numéricas:", num_cols)
print("Variables categóricas:", cat_cols)
print("X:", X.shape, "| y:", y.shape)

In [ ]:
num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numericas", num_pipeline, num_cols),
    ("categoricas", cat_pipeline, cat_cols)
])

X_preprocesado = preprocessor.fit_transform(X)
print("Matriz preprocesada creada correctamente")
print("Dimensiones:", X_preprocesado.shape)

In [ ]:
# Recuperación de nombres de columnas transformadas para inspección
nombres_cat = preprocessor.named_transformers_["categoricas"].named_steps["encoder"].get_feature_names_out(cat_cols)
nombres_finales = num_cols + list(nombres_cat)

X_preprocesado_df = pd.DataFrame(
    X_preprocesado.toarray() if hasattr(X_preprocesado, "toarray") else X_preprocesado,
    columns=nombres_finales
)

X_preprocesado_df.head()

**Justificación del pipeline:**  
El pipeline evita aplicar transformaciones de manera manual y desordenada. Además, permite que imputación, escalado y codificación se ejecuten siempre en el mismo orden, reduciendo errores y dejando el flujo listo para modelos predictivos posteriores como Logistic Regression, Decision Tree o SVM.

## 15. Exportación de datos limpios y reporte reproducible

Se guardan archivos resultantes para que el análisis sea reproducible y pueda revisarse posteriormente.

In [ ]:
df_final.to_csv("dataset_clientes_limpio_transformado.csv", index=False)
X_preprocesado_df.to_csv("dataset_clientes_preprocesado_modelo.csv", index=False)

print("Archivos exportados:")
print("- dataset_clientes_limpio_transformado.csv")
print("- dataset_clientes_preprocesado_modelo.csv")

## 16. Documentación de entorno

Para cumplir reproducibilidad, se genera un archivo `requirements.txt` con las librerías principales utilizadas.

In [ ]:
requirements = """pandas
numpy
matplotlib
seaborn
scikit-learn
jupyter
"""

with open("requirements.txt", "w", encoding="utf-8") as archivo:
    archivo.write(requirements)

print("Archivo requirements.txt creado correctamente")
print(requirements)

## 17. Conclusiones finales

1. El dataset presentaba problemas de calidad relevantes: duplicados exactos, nulos en variables económicas y de score, valores negativos inválidos y outliers en variables financieras.
2. La limpieza se realizó con criterios técnicos: duplicados exactos eliminados, negativos convertidos a nulos, imputación con mediana y winsorización de outliers por IQR.
3. Se incorporó ingeniería de características con sentido de negocio, especialmente ratios financieros y variables de actividad del cliente.
4. Se aplicaron filtros avanzados, agrupaciones múltiples y joins para fortalecer el análisis exploratorio.
5. Las visualizaciones fueron acompañadas de interpretación, corrigiendo la ausencia de comunicación analítica observada en la retroalimentación.
6. Se dejó un pipeline de preprocesamiento reproducible para futuros modelos predictivos, alineado con buenas prácticas de Data Science.